# 04 — Feature Selection

**Primary:** Zhilin Zhang  
**Support:** Tianyi Qin

Requirement:
- one **embedded** method;
- one **filter** method;
- top 3 features from each;
- one specific hard-case listing with actual values;
- explanation of why methods may rank features differently.

## 1. Imports and data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif

candidates = [Path("../data/processed_listings.csv"), Path("data/processed_listings.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Create data/processed_listings.csv first.")

df = pd.read_csv(DATA_PATH)
print(df.shape)

## 2. Reuse the exact modelling split/features

Do not independently create a different target or different rows here. The feature-selection ranking must be traceable to the group's modelling data.

Recommended development approach:
- save row IDs / split assignments from the modelling workflow;
- recreate the same training rows here;
- use the exact full feature set used in the size+location+amenities model.

In [ ]:
ID_COL = "id"
TARGET = "high_price"

if ID_COL not in df.columns:
    raise KeyError("Preserve listing id during preprocessing for the hard-case requirement.")

# TODO: load/recreate the same train split and target threshold used in 03_modelling.ipynb.

## 3. Embedded method

Use the fitted Decision Tree's feature importances after preprocessing. If one-hot encoding expands categorical variables, decide and document whether importances are reported at encoded-column level or aggregated back to the original feature.

In [ ]:
# Example after obtaining the final fitted tree and transformed feature names:
#
# tree = experiments[("DecisionTree", "size_location_amenities")]["search"].best_estimator_
# prep = tree.named_steps["prep"]
# model = tree.named_steps["model"]
# feature_names = prep.get_feature_names_out()
#
# embedded = pd.Series(
#     model.feature_importances_,
#     index=feature_names
# ).sort_values(ascending=False)
#
# embedded_top3 = embedded.head(3)
# display(embedded_top3)

## 4. Filter method

Mutual Information is a suitable filter option if discrete/continuous handling is explicitly justified. Use training data only.

In [ ]:
# Example after producing the same transformed X_train matrix:
#
# X_train_transformed = prep.fit_transform(X_train)
# feature_names = prep.get_feature_names_out()
#
# mi = mutual_info_classif(
#     X_train_transformed,
#     y_train,
#     random_state=42,
# )
#
# filter_scores = pd.Series(mi, index=feature_names).sort_values(ascending=False)
# filter_top3 = filter_scores.head(3)
# display(filter_top3)

## 5. Compare top 3 lists

Preserve **feature names and actual scores** from both methods.

In [ ]:
comparison = pd.DataFrame({
    "embedded_feature": [],
    "embedded_score": [],
    "filter_feature": [],
    "filter_score": [],
})
comparison

## 6. Hard-case listing

Rubric: identify one specific listing by ID where:
- the two methods' top-ranked features would lead to different predictions, **or**
- the listing is genuinely far outside the typical range of the top-ranked feature.

The IQR scaffold below provides a reproducible route for a numeric top-ranked feature.

In [ ]:
def numeric_outlier_candidates(frame, feature, id_col="id"):
    x = pd.to_numeric(frame[feature], errors="coerce")
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    mask = x.lt(low) | x.gt(high)
    cols = [id_col, feature]
    return frame.loc[mask, cols].assign(
        lower_bound=low,
        upper_bound=high,
    )

# Example:
# hard_cases = numeric_outlier_candidates(df, "amenity_count")
# display(hard_cases.head(20))

## 7. Evidence checklist

Before finalising:
- [ ] embedded top 3 + scores;
- [ ] filter top 3 + scores;
- [ ] actual disagreement explained using the group's rankings;
- [ ] hard-case listing ID;
- [ ] relevant actual attribute values for that listing;
- [ ] concrete feature-set-specific scenario for opposite ranking behaviour;
- [ ] limitations specific to the two selected methods.